# InsightCart — Hugging Face Sentiment Analysis Evaluation

This notebook tests sentiment analysis independently before integrating it into InsightCart.

**Model:** `cardiffnlp/twitter-roberta-base-sentiment-latest`  
**Labels:** negative, neutral, positive  
**Approach:** runs the public model locally with `transformers`; it does not call the hosted Hugging Face API.

> Important: sentiment is not review authenticity. A negative review can be genuine, and a positive review can be fake. Keep these as separate predictions.

This model was trained on social-media text, so treat it as a baseline and test it against real product reviews before deciding whether to use it in production.


## 1. Install dependencies

Run this cell in your notebook environment. The first model load requires internet access to download the model.

In [1]:
%pip install -q transformers torch pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 2. Load the pretrained model

In [2]:
import torch
import pandas as pd
from transformers import pipeline

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
device = 0 if torch.cuda.is_available() else -1

sentiment_model = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device,
    truncation=True
)

print("Loaded:", MODEL_NAME)
print("Device:", "GPU" if device == 0 else "CPU")

/Users/hetlimbani/Desktop/Self_Learning/InsightCart/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1025 16:21:43.305000 30168 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 45923.04it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: cardiffnlp/twitter-roberta-base-sentiment-latest
Device: CPU


## 3. Test sample product reviews

Check whether the predictions make sense for positive, negative, neutral, and mixed reviews.

In [3]:
sample_reviews = [
    "This product is excellent. The quality is amazing and it works perfectly.",
    "Terrible product. It stopped working after one day and customer support was useless.",
    "The product arrived yesterday. It is black and comes with a charging cable.",
    "The build quality is good, but the battery drains very quickly.",
    "I love this purchase. It is worth every penny.",
    "Completely misleading. The product looks nothing like the pictures.",
    "It is okay for the price, but nothing special.",
    "Fast delivery and excellent packaging. I am very happy with this purchase."
]

predictions = sentiment_model(sample_reviews, batch_size=8)
results_df = pd.DataFrame([
    {"review": review, "sentiment": pred["label"].lower(),
     "confidence": round(float(pred["score"]), 4)}
    for review, pred in zip(sample_reviews, predictions)
])
display(results_df)

,review,sentiment,confidence
0,This product is excellent. The quality is amaz...,positive,0.9866
1,Terrible product. It stopped working after one...,negative,0.9577
2,The product arrived yesterday. It is black and...,neutral,0.4988
3,"The build quality is good, but the battery dra...",negative,0.5698
4,I love this purchase. It is worth every penny.,positive,0.9869
5,Completely misleading. The product looks nothi...,negative,0.9039
6,"It is okay for the price, but nothing special.",neutral,0.6220
7,Fast delivery and excellent packaging. I am ve...,positive,0.9874


## 4. Test your own review

Edit `my_review`, then run the cell.

In [4]:
my_review = "The camera quality is fantastic, but the battery life is disappointing."

if not my_review.strip():
    raise ValueError("Enter a review before running this cell.")

prediction = sentiment_model(my_review, truncation=True)[0]
print("Review:", my_review)
print("Predicted sentiment:", prediction["label"].lower())
print("Confidence:", round(float(prediction["score"]), 4))
print("Note: confidence is the model's score, not a guarantee that the prediction is correct.")

Review: The camera quality is fantastic, but the battery life is disappointing.
Predicted sentiment: negative
Confidence: 0.7276
Note: confidence is the model's score, not a guarantee that the prediction is correct.


## 5. Optional: Evaluate on manually labeled reviews

For a meaningful evaluation, replace these examples with a larger set of real product reviews that you have labeled yourself. This tiny sample is only a code sanity check, not a reliable performance benchmark.

In [5]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

labeled_reviews = [
    {"text": "Excellent sound quality and very easy to use.", "label": "positive"},
    {"text": "Stopped charging after two days. Very disappointed.", "label": "negative"},
    {"text": "The package contains the device and a USB cable.", "label": "neutral"},
    {"text": "Good price, but the screen scratches too easily.", "label": "negative"},
    {"text": "Works exactly as described. I am happy with it.", "label": "positive"},
    {"text": "It arrived on Monday in a cardboard box.", "label": "neutral"},
]

true_labels = [item["label"] for item in labeled_reviews]
test_predictions = sentiment_model([item["text"] for item in labeled_reviews], batch_size=8)
predicted_labels = [item["label"].lower() for item in test_predictions]

print("Accuracy:", round(accuracy_score(true_labels, predicted_labels), 4))
print("\nClassification report:")
print(classification_report(
    true_labels, predicted_labels,
    labels=["negative", "neutral", "positive"],
    zero_division=0
))
print("Confusion matrix (negative, neutral, positive):")
print(confusion_matrix(
    true_labels, predicted_labels,
    labels=["negative", "neutral", "positive"]
))

Accuracy: 1.0

Classification report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00         2
     neutral       1.00      1.00      1.00         2
    positive       1.00      1.00      1.00         2

    accuracy                           1.00         6
   macro avg       1.00      1.00      1.00         6
weighted avg       1.00      1.00      1.00         6

Confusion matrix (negative, neutral, positive):
[[2 0 0]
 [0 2 0]
 [0 0 2]]


## 6. If you specifically want the hosted Hugging Face API

The notebook above uses a downloaded local model. For hosted inference, Hugging Face documents `InferenceClient` and requires a user token with appropriate Inference Providers permissions. Keep that token in an environment variable (for example `HF_TOKEN`) and never commit it to GitHub.

After you assess quality, the hosted API can be tested separately. API/provider availability, rate limits, and billing can vary, so verify the current model/provider support before production integration.

## 7. Integration checklist

- Evaluate a meaningful sample of real product reviews from your target categories.
- Manually label sentiment as negative, neutral, or positive.
- Inspect per-class precision/recall and mixed/sarcastic review errors.
- Return sentiment separately from the existing fake/genuine prediction.
- Do not treat negative sentiment as evidence that a review is fake.
